In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# compare model's hyperparameters

In [2]:
def compare_plot(df, col_x, col_valilla, col_jimmy, y_label, title=""):
    x = df[col_x]
    y_vanilla = df[col_valilla]
    y_jimmy = df[col_jimmy]
    y_all = df[[col_valilla, col_jimmy]]

    fig = plt.figure(figsize=(12,6))
    plt.title(title)

    plt.plot(x, y_vanilla, zorder=3, markeredgecolor="black", alpha=0.25)
    plt.plot(x, y_jimmy, zorder=3, markeredgecolor="black", alpha=0.25)

    plt.scatter(x, y_vanilla, zorder=3, label="vanillaNAS")
    plt.scatter(x, y_jimmy, zorder=3, label="FCNNAS")

    width = 0.3
    xmin = [i - width/2 for i in range(len(x))]
    xmax = [i + width/2 for i in range(len(x))]

    plt.hlines(y=y_vanilla, xmin=xmin, xmax=xmax, color='black', linewidth=1, zorder=2)
    plt.hlines(y=y_jimmy, xmin=xmin, xmax=xmax, color='black', linewidth=1, zorder=2)

    plt.grid(axis="x",zorder=2)
    plt.grid(axis="y",zorder=2, linestyle= (0, (2, 10)))

    plt.xticks(rotation=15)
    plt.xlabel("Hyperparameter (K: Kernel, C: Cell)")
    plt.ylabel(y_label)
    plt.legend()

    return fig
    


In [3]:
dataset_paths = [path for path in Path("../Experiments_Compare_Model").iterdir() if path.is_dir()]
log_filname = "model_log.csv"

In [4]:
visualize_pic_path = Path("visualize_result")
visualize_pic_path.mkdir(parents=True, exist_ok=True)

In [5]:
metrices = {
        "best_acc": {"y_label":"Accuracy"}, 
        "tflite_acc": {"y_label":"Accuracy"}
        }
col_x = "decision_variable"
vanilla_name = "vanillaNAS_dense_"
jimmy_name = "JimmyNAS_I_fullyCNN_"
for dataset_path in dataset_paths:
    output_path = (visualize_pic_path/dataset_path.name)
    output_path.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(dataset_path/log_filname)
    print(dataset_path.name)
    temp_dict = {metric:{"vanilla":vanilla_name+metric, "jimmy":jimmy_name+metric}for metric in metrices.keys()}
    for key, value in temp_dict.items():
        fig = compare_plot(df, col_x, value["vanilla"], value["jimmy"], metrices[key]["y_label"])
        fig.savefig(output_path/f"{key}.png", dpi=300, bbox_inches='tight')
        plt.close(fig)

Animals-3
Flowers-4
MelanomaSkinCancer
MNIST
visual_wake_words


In [6]:
col_x = "decision_variable"
metrices = {
    "macs": {"y_label":"MACC"},
    "flash": {"y_label":"Flash (Byte)"}, 
    "peak_ram": {"y_label":"Peak RAM (Byte)"}, 
    "param_count": {"y_label":"Parameters"}
    }
vanilla_name = "vanillaNAS_dense_"
jimmy_name = "JimmyNAS_I_fullyCNN_"
for dataset_path in dataset_paths:
    output_path.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(dataset_path/log_filname)
    print(dataset_path.name)
    temp_dict = {metric:{"vanilla":vanilla_name+metric, "jimmy":jimmy_name+metric}for metric in metrices.keys()}
    for key, value in temp_dict.items():
        fig = compare_plot(df, col_x, value["vanilla"], value["jimmy"], metrices[key]["y_label"])
        fig.savefig(visualize_pic_path/f"{key}.png", dpi=300, bbox_inches='tight')
        plt.close(fig)
        
    break # use only 1 dataset to get model properties

Animals-3


# compare model kfold

In [ ]:
# def kfold_plot(df, col_x, col_valilla, col_jimmy, y_label, title=""):
#     x = df[col_x]
#     y_vanilla = df[col_valilla]
#     y_jimmy = df[col_jimmy]
#     y_all = df[[col_valilla, col_jimmy]]

#     fig = plt.figure(figsize=(12,6))
#     plt.title(title)

#     # plt.plot(x, y_vanilla, zorder=3, markeredgecolor="black", alpha=0.25)
#     # plt.plot(x, y_jimmy, zorder=3, markeredgecolor="black", alpha=0.25)

#     plt.scatter(x, y_vanilla, zorder=3, label="vanillaNAS")
#     plt.scatter(x, y_jimmy, zorder=3, label="JimmyNAS")

#     width = 0.3
#     xmin = [i - width/2 for i in range(len(x))]
#     xmax = [i + width/2 for i in range(len(x))]

#     plt.hlines(y=y_vanilla, xmin=xmin, xmax=xmax, color='black', linewidth=1, zorder=2)
#     plt.hlines(y=y_jimmy, xmin=xmin, xmax=xmax, color='black', linewidth=1, zorder=2)

#     plt.grid(axis="x",zorder=2)
#     plt.grid(axis="y",zorder=2, linestyle= (0, (2, 10)))

#     plt.xticks(rotation=15)
#     plt.xlabel("Hyperparameter (K: Kernel, C: Cell)")
#     plt.ylabel(y_label)
#     plt.legend()

#     return fig
    


In [14]:

def kfold_plot(df, col_x, col_valilla, col_jimmy, y_label, metric, title=""):
    fig = plt.figure(figsize=(12,6))
    plt.title(title)
    
    df = df[[col_x, col_valilla, col_jimmy]]
    df_temp = df[["k-fold"]].merge(pd.DataFrame(df.iloc[:,1:].T.index), how="cross")
    df_temp.rename(columns={0:"model"}, inplace=True)
    df_temp["model"] = df_temp["model"].str.split(metric).apply(lambda x : x[0][:-1])
    df_temp[metric] = pd.NA
    df_temp.loc[df_temp["model"]=="vanillaNAS_dense", [metric]] = df[col_valilla].values
    df_temp.loc[df_temp["model"]=="JimmyNAS_I_fullyCNN", [metric]] = df[col_jimmy].values
    df_temp.loc[df_temp["model"]=="vanillaNAS_dense", "model"] = "vanillaNAS"
    df_temp.loc[df_temp["model"]=="JimmyNAS_I_fullyCNN", "model"] = "FCNNAS"
    # One line to rule them all
    ax = sns.barplot(data=df_temp, x=col_x, y=metric, hue='model')

    # Iterate through the containers and add labels
    for container in ax.containers:
        ax.bar_label(container, padding=3)

    offset = (df_temp[metric].max() - df_temp[metric].min())*0.2
    plt.ylim(df_temp[metric].min()-offset, df_temp[metric].max()+offset)
    plt.grid(axis="y", ls="-", alpha=0.5)
    plt.ylabel(y_label)
    return fig
    


In [9]:
dataset_paths = [path for path in Path("../Experiments_KFolds2").iterdir() if path.is_dir()]
log_filname = "models_log.csv"

In [10]:
visualize_pic_path = Path("visualize_result")
visualize_pic_path.mkdir(parents=True, exist_ok=True)

In [15]:

metrices = {
        "best_acc": {"y_label":"Accuracy"}, 
        "tflite_acc": {"y_label":"Accuracy"},
        "tflite_precision": {"y_label":"Precision"},
        "tflite_recall": {"y_label":"Recall"},
        "tflite_f1": {"y_label":"F1-score"},
        "param_count": {"y_label":"Parameters"},
        "mac_count": {"y_label":"MACC"},
        "flash": {"y_label":"Flash (Byte)"},
        "peak_ram": {"y_label":"Peak RAM(Byte)"},
        }
col_x = "k-fold"
vanilla_name = "vanillaNAS_dense_"
jimmy_name = "JimmyNAS_I_fullyCNN_"
for dataset_path in dataset_paths:
    output_path = (visualize_pic_path/Path(dataset_path.name)/"kfold")
    output_path.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(dataset_path/log_filname)
    print(dataset_path.name)
    temp_dict = {metric:{"vanilla":vanilla_name+metric, "jimmy":jimmy_name+metric} for metric in metrices.keys()}
    for key, value in temp_dict.items():
        fig = kfold_plot(df, col_x, value["vanilla"], value["jimmy"], metrices[key]["y_label"], metric=key)
        fig.savefig(output_path/f"{key}.png", dpi=300, bbox_inches='tight')
        plt.close(fig)

Animals-3
Flowers-4
MelanomaSkinCancer
MNIST
visual_wake_words
